### Statistical Analysis

#### Project objective

This notebook uses the training data to examine whether four transaction features are associated with fraud. It builds on patterns noticed during exploratory data analysis and applies statistical tests and measures to describe those relationships.

For each feature, we compare fraud rates between groups, use a Chi-square test of independence, and calculate Cramér's V, a Risk Ratio, an Odds Ratio, and confidence intervals for the relative measures. The results describe unadjusted associations in this dataset; they do not show cause and effect or prove that a feature will improve a predictive model.

The four features are CVV retry count, foreign transaction status, IP country mismatch, and billing/shipping mismatch.


The Chi-square test also assumes that observations are independent. This is a property of how the data were collected and structured; checking expected cell counts alone does not verify it.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency


In [6]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Load data
train_path = RAW_DATA_DIR / "train.csv"
train_df = pd.read_csv(train_path)

print(f"Training data shape: {train_df.shape}")

Training data shape: (699960, 26)


#### Investigation 1: CVV retry count and fraud

#### Step 1: Prepare the comparison groups

**Question:** Do transactions with at least one CVV retry have a different fraud rate from transactions with no retries?

We group `cvv_retry_count` into zero retries and one or more retries, and compare each group's fraud outcome. The notebook copies only the needed columns into a separate DataFrame, so the original training data is not changed.

- **H₀:** CVV retry group and fraud outcome are independent (there is no association).
- **H₁:** CVV retry group and fraud outcome are associated.


In [7]:
# Create a separate DataFrame for this investigation
cvv_analysis_df = train_df[["cvv_retry_count", "is_fraud"]].copy()

# Group transactions by CVV retries
cvv_analysis_df["cvv_retry_group"] = (
    cvv_analysis_df["cvv_retry_count"]
    .gt(0)
    .map({
        False: "Zero retries",
        True: "One or more retries"
    })
)

# Summarise each group
cvv_summary = (
    cvv_analysis_df
    .groupby("cvv_retry_group")
    .agg(
        total_transactions=("is_fraud", "count"),
        fraudulent_transactions=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
)

# Convert fraud rate to percentage
cvv_summary["fraud_rate"] *= 100

cvv_summary

,total_transactions,fraudulent_transactions,fraud_rate
cvv_retry_group,,,
One or more retries,57872,5699.0,9.847595
Zero retries,642088,5990.0,0.932894


#### Step 2: Compare fraud rates

The summary shows each group's transaction count, fraudulent transaction count, and fraud rate (fraudulent transactions divided by all transactions in that group).

- **One or more retries:** 5,699 of 57,872 transactions; **9.85%** fraud rate.
- **Zero retries:** 5,990 of 642,088 transactions; **0.93%** fraud rate.

The rates differ descriptively. The test below checks whether the pattern is consistent with an association in the data.


#### Step 3: Contingency table

This table counts legitimate and fraudulent transactions for each retry group. These observed counts are the input to the Chi-square test.


In [8]:
contingency_table = pd.crosstab(
    cvv_analysis_df["cvv_retry_group"],
    cvv_analysis_df["is_fraud"]
)

# Rename outcome columns
contingency_table.columns = ["Legitimate", "Fraudulent"]

contingency_table

,Legitimate,Fraudulent
cvv_retry_group,,
One or more retries,52173,5699
Zero retries,636098,5990


#### Step 4: Chi-square test — method

This test compares the observed counts with counts expected if retry group and fraud outcome were independent. It reports the test statistic, p-value, degrees of freedom, and expected counts.


In [10]:

chi2_stat, p_value, degrees_of_freedom, expected_counts = (
    chi2_contingency(contingency_table)
)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {degrees_of_freedom}")

print("\nExpected counts:")
print(
    pd.DataFrame(
        expected_counts,
        index=contingency_table.index,
        columns=contingency_table.columns
    )
)

Chi-square statistic: 25687.4625
P-value: 0.0000000000
Degrees of freedom: 1

Expected counts:
                        Legitimate    Fraudulent
cvv_retry_group                                 
One or more retries   56905.565049    966.434951
Zero retries         631365.434951  10722.565049


#### Chi-square test — results and interpretation

This test checks whether retry group and fraud outcome are associated by comparing the observed table counts with the counts expected if the variables were independent. The expected counts are all above 5, meeting the usual condition for this Chi-square approximation.

**Results:** χ² = **25,687.4625**, degrees of freedom = **1**. The p-value prints as **0.0000000000** to ten decimal places; it is very small, not exactly zero. At the 0.05 significance level, we reject the null hypothesis and find evidence of an association. The test indicates evidence, while the measures below describe its size.


#### Step 5: Measure the association with Cramér's V

The Chi-square test evaluates whether there is evidence of an association, but its result does not describe the strength of that association. Cramér's V provides an effect-size measure for the relationship between the two categorical variables. Its value ranges from 0 to 1, with values closer to 0 indicating a weaker association and values closer to 1 indicating a stronger association.

In [12]:
# Calculate Cramér's V
n = contingency_table.to_numpy().sum()

minimum_dimension = min(contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2_stat / (n * minimum_dimension)
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.1916


#### Step 6: Risk Ratio

The Risk Ratio divides the fraud rate in the one-or-more-retries group by the rate in the zero-retries group. A value above 1 means the first group's observed risk is higher.


In [13]:
# Extract fraud rates
retry_fraud_rate = (
    contingency_table.loc["One or more retries", "Fraudulent"]
    / contingency_table.loc["One or more retries"].sum()
)

zero_retry_fraud_rate = (
    contingency_table.loc["Zero retries", "Fraudulent"]
    / contingency_table.loc["Zero retries"].sum()
)

# Calculate risk ratio
risk_ratio = retry_fraud_rate / zero_retry_fraud_rate

print(f"Risk Ratio: {risk_ratio:.4f}")

Risk Ratio: 10.5560


#### Step 7: Odds Ratio

Odds compare fraudulent transactions with legitimate transactions within a group. The Odds Ratio compares those odds for the two retry groups; a value above 1 means higher odds in the one-or-more-retries group.


In [15]:
a = contingency_table.loc["One or more retries", "Fraudulent"]
b = contingency_table.loc["One or more retries", "Legitimate"]

c = contingency_table.loc["Zero retries", "Fraudulent"]
d = contingency_table.loc["Zero retries", "Legitimate"]

# Calculate odds for each group
retry_odds = a / b
zero_retry_odds = c / d

# Calculate odds ratio
odds_ratio = retry_odds / zero_retry_odds

print(f"Retry group odds: {retry_odds:.4f}")
print(f"Zero-retry group odds: {zero_retry_odds:.4f}")
print(f"Odds Ratio: {odds_ratio:.4f}")

Retry group odds: 0.1092
Zero-retry group odds: 0.0094
Odds Ratio: 11.5998


#### Step 8: Confidence intervals

The 95% confidence intervals give ranges of plausible values for the estimated Risk Ratio and Odds Ratio under the calculation assumptions. An interval that excludes 1 is consistent with a relative difference between groups.


In [16]:
# Risk ratio confidence interval
se_log_rr = np.sqrt((1 / a) - (1 / (a + b)) + (1 / c) - (1 / (c + d)))

rr_lower = np.exp(np.log(risk_ratio) - 1.96 * se_log_rr)
rr_upper = np.exp(np.log(risk_ratio) + 1.96 * se_log_rr)

# Odds ratio confidence interval
se_log_or = np.sqrt((1 / a) + (1 / b) + (1 / c) + (1 / d))

or_lower = np.exp(np.log(odds_ratio) - 1.96 * se_log_or)
or_upper = np.exp(np.log(odds_ratio) + 1.96 * se_log_or)

print(f"Risk Ratio: {risk_ratio:.4f}")
print(f"95% CI: [{rr_lower:.4f}, {rr_upper:.4f}]")

print(f"\nOdds Ratio: {odds_ratio:.4f}")
print(f"95% CI: [{or_lower:.4f}, {or_upper:.4f}]")

Risk Ratio: 10.5560
95% CI: [10.1903, 10.9348]

Odds Ratio: 11.5998
95% CI: [11.1745, 12.0412]


#### CVV retry count: Results and conclusion

The observed fraud rates were **9.85%** for one or more retries and **0.93%** for zero retries. The estimated Risk Ratio was **10.5560** (95% CI: **10.1903–10.9348**), and the Odds Ratio was **11.5998** (95% CI: **11.1745–12.0412**). Both confidence intervals are above 1. Cramér's V was **0.1916**. These measures use different scales: relative risk compares group fraud rates, while Cramér's V summarizes association across the table.

**Conclusion:** Retry group is associated with fraud in this training data. The rate difference and relative measures make it a candidate feature to evaluate in later modelling; they do not establish cause and effect or guarantee predictive value.


#### Investigation 2: Foreign transactions and fraud

**Question:** Is the observed fraud rate different for foreign and domestic transactions?

Domestic transactions are the reference group for the Risk Ratio and Odds Ratio. We compare group summaries, test the association, and quantify it.

- **H₀:** Foreign transaction status and fraud outcome are independent.
- **H₁:** Foreign transaction status and fraud outcome are associated.


#### Step 1: Compare domestic and foreign transactions

The summary reports transaction counts, fraudulent counts, and fraud rates for each group. Rates use each group's own total as the denominator.


In [17]:
foreign_analysis_df = train_df[
    ["is_foreign_transaction", "is_fraud"]
].copy()

foreign_summary = (
    foreign_analysis_df
    .groupby("is_foreign_transaction")
    .agg(
        total_transactions=("is_fraud", "count"),
        fraudulent_transactions=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
)

foreign_summary["fraud_rate"] *= 100

foreign_summary

,total_transactions,fraudulent_transactions,fraud_rate
is_foreign_transaction,,,
False,687862,10168.0,1.478203
True,12098,1521.0,12.572326


#### Step 2: Contingency table

The table counts legitimate and fraudulent transactions in the domestic and foreign groups. These counts are used in the test below.


In [18]:
foreign_contingency_table = pd.crosstab(
    foreign_analysis_df["is_foreign_transaction"],
    foreign_analysis_df["is_fraud"]
)

foreign_contingency_table.columns = ["Legitimate", "Fraudulent"]

foreign_contingency_table.index = ["Domestic", "Foreign"]

foreign_contingency_table

,Legitimate,Fraudulent
Domestic,677694,10168
Foreign,10577,1521


#### Step 3: Chi-square test of independence

The test compares observed counts with counts expected if foreign transaction status and fraud outcome were independent. The displayed expected counts are all above 5.


In [19]:
chi2_stat, p_value, degrees_of_freedom, expected_counts = chi2_contingency(
    foreign_contingency_table
)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {degrees_of_freedom}")

print("\nExpected counts:")
pd.DataFrame(
    expected_counts,
    index=foreign_contingency_table.index,
    columns=foreign_contingency_table.columns
)

Chi-square statistic: 8904.4659
P-value: 0.0000000000
Degrees of freedom: 1

Expected counts:


,Legitimate,Fraudulent
Domestic,676375.030862,11486.969138
Foreign,11895.969138,202.030862


#### Step 4: Cramér's V

Cramér's V summarizes the strength of association between foreign status and fraud outcome.


In [20]:
n = foreign_contingency_table.to_numpy().sum()

minimum_dimension = min(foreign_contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2_stat / (n * minimum_dimension)
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.1128


#### Step 5: Risk Ratio

The Risk Ratio compares the foreign transaction fraud rate with the domestic rate, using domestic transactions as the reference group.


In [22]:
a = foreign_contingency_table.loc["Foreign", "Fraudulent"]
b = foreign_contingency_table.loc["Foreign", "Legitimate"]

c = foreign_contingency_table.loc["Domestic", "Fraudulent"]
d = foreign_contingency_table.loc["Domestic", "Legitimate"]

foreign_fraud_risk = a / (a + b)
domestic_fraud_risk = c / (c + d)

risk_ratio = foreign_fraud_risk / domestic_fraud_risk

print(f"Foreign fraud risk: {foreign_fraud_risk:.4%}")
print(f"Domestic fraud risk: {domestic_fraud_risk:.4%}")
print(f"Risk Ratio: {risk_ratio:.4f}")

Foreign fraud risk: 12.5723%
Domestic fraud risk: 1.4782%
Risk Ratio: 8.5051


#### Step 6: Odds Ratio

The Odds Ratio compares fraud odds for foreign and domestic transactions. It measures odds, which differ from the fraud probabilities summarized by the Risk Ratio.


In [23]:
foreign_odds = a / b
domestic_odds = c / d

odds_ratio = foreign_odds / domestic_odds

print(f"Foreign transaction odds: {foreign_odds:.4f}")
print(f"Domestic transaction odds: {domestic_odds:.4f}")
print(f"Odds Ratio: {odds_ratio:.4f}")

Foreign transaction odds: 0.1438
Domestic transaction odds: 0.0150
Odds Ratio: 9.5844


#### Step 7: Confidence intervals

The 95% confidence intervals show uncertainty around the estimated relative measures. Values above 1 throughout an interval are consistent with a higher rate or odds in the foreign group.


In [24]:
# Risk ratio confidence interval
se_log_rr = np.sqrt(
    (1 / a) - (1 / (a + b)) +
    (1 / c) - (1 / (c + d))
)

rr_lower = np.exp(np.log(risk_ratio) - 1.96 * se_log_rr)
rr_upper = np.exp(np.log(risk_ratio) + 1.96 * se_log_rr)


# Odds ratio confidence interval
se_log_or = np.sqrt(
    (1 / a) + (1 / b) + (1 / c) + (1 / d)
)

or_lower = np.exp(np.log(odds_ratio) - 1.96 * se_log_or)
or_upper = np.exp(np.log(odds_ratio) + 1.96 * se_log_or)


print(f"Risk Ratio: {risk_ratio:.4f}")
print(f"95% CI: [{rr_lower:.4f}, {rr_upper:.4f}]")

print(f"\nOdds Ratio: {odds_ratio:.4f}")
print(f"95% CI: [{or_lower:.4f}, {or_upper:.4f}]")

Risk Ratio: 8.5051
95% CI: [8.0839, 8.9483]

Odds Ratio: 9.5844
95% CI: [9.0515, 10.1487]


#### Foreign transactions: Results and conclusion

The observed fraud rate was **12.57%** among foreign transactions (1,521 of 12,098) and **1.48%** among domestic transactions (10,168 of 687,862). The Chi-square result was χ² = **8,904.4659**, df = **1**; the p-value displays as **0.0000000000** to ten decimal places, which is a rounded value. All expected counts exceed 5, so at the 0.05 level we reject independence.

The Risk Ratio was **8.5051** (95% CI: **8.0839–8.9483**), the Odds Ratio was **9.5844** (95% CI: **9.0515–10.1487**), and Cramér's V was **0.1128**.

**Conclusion:** Foreign transaction status is associated with a higher observed fraud rate in this training data. It is a candidate feature for later evaluation. This unadjusted association does not establish causation or predictive usefulness.


#### Investigation 3: IP country mismatch and fraud

**Question:** Is fraud more common when the transaction's IP country does not match the expected country?

The **False** group is no mismatch and serves as the reference group; **True** means a mismatch.

- **H₀:** IP country mismatch status and fraud outcome are independent.
- **H₁:** IP country mismatch status and fraud outcome are associated.


#### Step 1: Compare the groups

The summary reports group totals, fraudulent counts, and fraud rates for transactions with and without an IP country mismatch.


In [25]:
feature = "ip_country_mismatch"

ip_analysis_df = train_df[[feature, "is_fraud"]].copy()

ip_summary = (
    ip_analysis_df.groupby(feature)
    .agg(
        total_transactions=("is_fraud", "count"),
        fraudulent_transactions=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
)

ip_summary["fraud_rate"] *= 100

ip_summary

,total_transactions,fraudulent_transactions,fraud_rate
ip_country_mismatch,,,
False,688648,9823.0,1.426418
True,11312,1866.0,16.495757


#### Step 2: Contingency table

The table shows legitimate and fraudulent transaction counts in the mismatch and no-mismatch groups.


In [26]:
ip_contingency_table = pd.crosstab(
    ip_analysis_df[feature],
    ip_analysis_df["is_fraud"]
)

ip_contingency_table.columns = ["Legitimate", "Fraudulent"]
ip_contingency_table.index = ["No mismatch", "Mismatch"]

ip_contingency_table

,Legitimate,Fraudulent
No mismatch,678825,9823
Mismatch,9446,1866


#### Step 3: Chi-square test of independence

The test checks whether mismatch status and fraud outcome are associated by comparing observed counts with expected counts under independence. All displayed expected counts exceed 5.


In [28]:
chi2_stat, p_value, degrees_of_freedom, expected_counts = chi2_contingency(
    ip_contingency_table
)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {degrees_of_freedom}")

print("\nExpected counts:")
pd.DataFrame(
    expected_counts,
    index=ip_contingency_table.index,
    columns=ip_contingency_table.columns
)

Chi-square statistic: 15381.6337
P-value: 0.0000000000
Degrees of freedom: 1

Expected counts:


,Legitimate,Fraudulent
No mismatch,677147.905035,11500.094965
Mismatch,11123.094965,188.905035


#### Step 4: Effect sizes and relative measures

This cell calculates Cramér's V, the Risk Ratio, and the Odds Ratio. Cramér's V summarizes association; the relative measures compare fraud in the mismatch group with the no-mismatch reference group.


In [29]:
# Cramér's V
n = ip_contingency_table.to_numpy().sum()
minimum_dimension = min(ip_contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2_stat / (n * minimum_dimension)
)

# Define contingency table values
a = ip_contingency_table.loc["Mismatch", "Fraudulent"]
b = ip_contingency_table.loc["Mismatch", "Legitimate"]
c = ip_contingency_table.loc["No mismatch", "Fraudulent"]
d = ip_contingency_table.loc["No mismatch", "Legitimate"]

# Risk ratio
mismatch_risk = a / (a + b)
no_mismatch_risk = c / (c + d)
risk_ratio = mismatch_risk / no_mismatch_risk

# Odds ratio
mismatch_odds = a / b
no_mismatch_odds = c / d
odds_ratio = mismatch_odds / no_mismatch_odds

print(f"Cramér's V: {cramers_v:.4f}")
print(f"Risk Ratio: {risk_ratio:.4f}")
print(f"Odds Ratio: {odds_ratio:.4f}")

Cramér's V: 0.1482
Risk Ratio: 11.5645
Odds Ratio: 13.6514


#### Step 5: Confidence intervals

The intervals quantify uncertainty around the Risk Ratio and Odds Ratio estimates. An interval excluding 1 supports a relative difference between the groups.


In [30]:
# Risk ratio confidence interval
se_log_rr = np.sqrt(
    (1 / a) - (1 / (a + b)) +
    (1 / c) - (1 / (c + d))
)

rr_lower = np.exp(np.log(risk_ratio) - 1.96 * se_log_rr)
rr_upper = np.exp(np.log(risk_ratio) + 1.96 * se_log_rr)

# Odds ratio confidence interval
se_log_or = np.sqrt(
    (1 / a) + (1 / b) + (1 / c) + (1 / d)
)

or_lower = np.exp(np.log(odds_ratio) - 1.96 * se_log_or)
or_upper = np.exp(np.log(odds_ratio) + 1.96 * se_log_or)

print(f"RR 95% CI: [{rr_lower:.4f}, {rr_upper:.4f}]")
print(f"OR 95% CI: [{or_lower:.4f}, {or_upper:.4f}]")

RR 95% CI: [11.0459, 12.1074]
OR 95% CI: [12.9403, 14.4016]


#### IP country mismatch: Results and conclusion

The observed fraud rate was **16.50%** for transactions with a mismatch (1,866 of 11,312) and **1.43%** without one (9,823 of 688,648). The Chi-square result was χ² = **15,381.6337**, df = **1**; the p-value displays as **0.0000000000** to ten decimal places, not exactly zero. All expected counts exceed 5, so at the 0.05 level we reject independence.

The Risk Ratio was **11.5645** (95% CI: **11.0459–12.1074**), the Odds Ratio was **13.6514** (95% CI: **12.9403–14.4016**), and Cramér's V was **0.1482**.

**Conclusion:** IP mismatch had the highest observed fraud rate and largest Risk Ratio among these four comparisons. It is a candidate feature for later evaluation. This is an unadjusted association and does not show that mismatch causes fraud.


#### Investigation 4: Billing/shipping mismatch and fraud

**Question:** Is the fraud rate different when billing and shipping details do not match?

Transactions without a mismatch are the reference group for the relative measures.

- **H₀:** Billing/shipping mismatch status and fraud outcome are independent.
- **H₁:** Billing/shipping mismatch status and fraud outcome are associated.


#### Step 1: Compare the groups

The summary reports transaction totals, fraudulent counts, and fraud rates for the mismatch and no-mismatch groups.


In [31]:
feature = "billing_shipping_mismatch"

billing_analysis_df = train_df[[feature, "is_fraud"]].copy()

billing_summary = (
    billing_analysis_df.groupby(feature)
    .agg(
        total_transactions=("is_fraud", "count"),
        fraudulent_transactions=("is_fraud", "sum"),
        fraud_rate=("is_fraud", "mean")
    )
)

billing_summary["fraud_rate"] *= 100

billing_summary

,total_transactions,fraudulent_transactions,fraud_rate
billing_shipping_mismatch,,,
False,690906,10513.0,1.521625
True,9054,1176.0,12.988734


#### Step 2: Contingency table

The table counts legitimate and fraudulent transactions for each billing/shipping mismatch group.


In [32]:
billing_contingency_table = pd.crosstab(
    billing_analysis_df[feature],
    billing_analysis_df["is_fraud"]
)

billing_contingency_table.columns = ["Legitimate", "Fraudulent"]
billing_contingency_table.index = ["No mismatch", "Mismatch"]

billing_contingency_table

,Legitimate,Fraudulent
No mismatch,680393,10513
Mismatch,7878,1176


#### Step 3: Chi-square test of independence

The test compares observed counts with counts expected under independence. All displayed expected counts exceed 5.


In [33]:
chi2_stat, p_value, degrees_of_freedom, expected_counts = chi2_contingency(
    billing_contingency_table
)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {degrees_of_freedom}")

print("\nExpected counts:")
pd.DataFrame(
    expected_counts,
    index=billing_contingency_table.index,
    columns=billing_contingency_table.columns
)

Chi-square statistic: 7149.5687
P-value: 0.0000000000
Degrees of freedom: 1

Expected counts:


,Legitimate,Fraudulent
No mismatch,679368.197506,11537.802494
Mismatch,8902.802494,151.197506


#### Step 4: Association measures and confidence intervals

This cell calculates Cramér's V and the Risk Ratio and Odds Ratio, with 95% confidence intervals for the relative measures. Values above 1 indicate higher observed fraud risk or odds in the mismatch group than in the reference group.


In [34]:
# Cramér's V
n = billing_contingency_table.to_numpy().sum()
minimum_dimension = min(billing_contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2_stat / (n * minimum_dimension)
)

# Define contingency table values
a = billing_contingency_table.loc["Mismatch", "Fraudulent"]
b = billing_contingency_table.loc["Mismatch", "Legitimate"]
c = billing_contingency_table.loc["No mismatch", "Fraudulent"]
d = billing_contingency_table.loc["No mismatch", "Legitimate"]

# Risk ratio
mismatch_risk = a / (a + b)
no_mismatch_risk = c / (c + d)
risk_ratio = mismatch_risk / no_mismatch_risk

# Odds ratio
mismatch_odds = a / b
no_mismatch_odds = c / d
odds_ratio = mismatch_odds / no_mismatch_odds

# Confidence intervals
se_log_rr = np.sqrt(
    (1 / a) - (1 / (a + b)) +
    (1 / c) - (1 / (c + d))
)

rr_lower = np.exp(np.log(risk_ratio) - 1.96 * se_log_rr)
rr_upper = np.exp(np.log(risk_ratio) + 1.96 * se_log_rr)

se_log_or = np.sqrt(
    (1 / a) + (1 / b) + (1 / c) + (1 / d)
)

or_lower = np.exp(np.log(odds_ratio) - 1.96 * se_log_or)
or_upper = np.exp(np.log(odds_ratio) + 1.96 * se_log_or)

print(f"Cramér's V: {cramers_v:.4f}")
print(f"Risk Ratio: {risk_ratio:.4f}")
print(f"RR 95% CI: [{rr_lower:.4f}, {rr_upper:.4f}]")
print(f"Odds Ratio: {odds_ratio:.4f}")
print(f"OR 95% CI: [{or_lower:.4f}, {or_upper:.4f}]")

Cramér's V: 0.1011
Risk Ratio: 8.5361
RR 95% CI: [8.0665, 9.0331]
Odds Ratio: 9.6611
OR 95% CI: [9.0600, 10.3019]


#### Billing/shipping mismatch: Results and conclusion

The observed fraud rate was **12.99%** for transactions with a mismatch (1,176 of 9,054) and **1.52%** without one (10,513 of 690,906). The Chi-square result was χ² = **7,149.5687**, df = **1**; the p-value displays as **0.0000000000** to ten decimal places, not exactly zero. All expected counts exceed 5, so at the 0.05 level we reject independence.

The Risk Ratio was **8.5361** (95% CI: **8.0665–9.0331**), the Odds Ratio was **9.6611** (95% CI: **9.0600–10.3019**), and Cramér's V was **0.1011**.

**Conclusion:** Billing/shipping mismatch is associated with a higher observed fraud rate in this training data and is a candidate feature to evaluate. The mismatch group is much smaller than the no-mismatch group. The comparison is unadjusted and does not establish cause and effect.


#### Overall Statistical Analysis Summary

The four Chi-square tests provide evidence of an association between each investigated feature and fraud outcome in this training dataset. In each case, the p-value is extremely small and prints as `0.0000000000` when rounded to ten decimal places; it is not exactly zero.

The table brings together the main descriptive and effect-size results. The exposed group is the group with the feature condition being investigated; the reference group is the comparison group.

| Feature | Fraud rate in exposed group | Fraud rate in reference group | Risk Ratio | Cramér's V |
|---|---:|---:|---:|---:|
| CVV retry count (one or more vs zero) | 9.85% | 0.93% | 10.5560 | 0.1916 |
| Foreign transaction (foreign vs domestic) | 12.57% | 1.48% | 8.5051 | 0.1128 |
| IP country mismatch (mismatch vs none) | 16.50% | 1.43% | 11.5645 | 0.1482 |
| Billing/shipping mismatch (mismatch vs none) | 12.99% | 1.52% | 8.5361 | 0.1011 |

**Main observations**

- IP country mismatch had the highest observed fraud rate and the largest Risk Ratio.
- CVV retry count had the highest Cramér's V among the four comparisons.
- These rankings differ because Risk Ratio compares relative fraud rates, while Cramér's V summarizes the strength of association across the contingency table.

**Conclusion**

All four features show statistically significant, unadjusted associations with fraud in this training dataset. They are reasonable candidates to carry forward for model development and validation. These results do not establish causation, account for the influence of other features, or prove that any feature will improve predictive performance. Their usefulness must be assessed using appropriate validation data and fraud-detection metrics.